# Module 3 : le module qui apprend

Un système qui apprend à partir d'exemples fait-il mieux que nos règles ?

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/secumail")
except ImportError:
    os.chdir(Path.cwd().parents[1])  # depuis workshop/<atelier>/

sys.path.insert(0, os.getcwd())

# Règles écrites ou points appris ?

| | Modules 1 et 2 | Module 3 |
|---|---|---|
| Qui choisit les indices ? | vous | les mots |
| Qui choisit les points ? | vous | un calcul |

Le module 3 ne comprend pas les mots. Il **compte**.

In [ ]:
%xmode Minimal
from src import pipeline
from src.email import mots_du_courriel, mots_du_texte

jouet = [
    ("spam", "Gagnez un iPhone gratuit, cliquez ici !"),
    ("spam", "Cadeau gratuit pour vous : profitez-en vite."),
    ("spam", "Votre essai gratuit expire demain, cliquez ici."),
    ("spam", "Votre cadeau vous attend, cliquez maintenant."),
    ("spam", "Offre limitée : soldes sur tout le magasin."),
    ("ham", "La réunion de demain est reportée à 14 h."),
    ("ham", "Rappel : réunion du club demain midi."),
    ("ham", "Votre facture est disponible dans votre espace client."),
    ("ham", "Les photos de la réunion sont en ligne, cliquez ici."),
    ("ham", "Votre sortie de demain est confirmée, apportez un lunch."),
]

## Un courriel devient un sac de mots

<img src="images/sac_de_mots.svg" alt="" style="width:620px">

In [ ]:
courriel = {
    "subject": "Offre GRATUITE",
    "message": "Gagnez sur concours-gratuit.top ! Gratuit !",
}
print(mots_du_courriel(courriel))

« gratuite » compte comme un autre mot que « gratuit ».

## Même sac ?

Ces deux phrases donnent-elles le même sac de mots ?

- « Le chat mange la souris »
- « La souris mange le chat »

In [ ]:
a = mots_du_courriel({"message": "Le chat mange la souris"})
b = mots_du_courriel({"message": "La souris mange le chat"})
print(a == b)

Même sac, sens opposé. L'ordre des mots n'est pas important.

## Dix exemples dont on connaît la réponse

<img src="images/boite_exemples.svg" alt="" style="width:820px">

Dans combien de spams apparaît « cadeau » ? Et dans combien de
courriels légitimes ?

## Compter, c'est une boucle

In [ ]:
def compter(mot):
    spams = 0
    legitimes = 0
    for label, texte in jouet:
        if mot in mots_du_texte(texte):
            if label == "spam":
                spams += 1
            else:
                legitimes += 1
    return spams, legitimes


print("cadeau :", compter("cadeau"))
print("facture :", compter("facture"))

`apprendre()` fait ce calcul pour **tous** les mots et **toutes** les
catégories à la fois.

## « cadeau » penche vers le spam. De combien ?

<img src="images/balance.svg" alt="" style="width:420px">

On a 5 spams et 5 courriels légitimes : on peut comparer les comptes.

Première idée : diviser. 2 spams ÷ 0 courriel légitime…

In [ ]:
print(2 / 0)

On ajoute 1 à chaque compte : (2 + 1) ÷ (0 + 1) = **3**.
Le **rapport** de « cadeau » vaut 3.

## Du rapport aux points

<img src="images/escalier.svg" alt="" style="width:800px">

Combien de points pour un rapport de 16 ? Et de 1/8 ?

+4 et −3. Chaque fois que le rapport double, on gagne un point.

## En Python : `math.log2`

In [ ]:
import math

for rapport in [1 / 8, 1 / 2, 1, 3, 16]:
    print(f"{rapport:>6g}  ->  {math.log2(rapport):+.1f} points")

Le **signe** dit de quel côté penche le mot. La **taille** dit de combien.

## Additionner les points

<img src="images/tir_a_la_corde.svg" alt="" style="width:860px">

## Avec les dix exemples

In [ ]:
def points(mot):
    spams, legitimes = compter(mot)
    return math.log2((spams + 1) / (legitimes + 1))


for mot in mots_du_texte("Offre : un cadeau pour vous"):
    print(f"{mot:<8} {points(mot):+.1f}")

Quel est le total ? SPAM ou LÉGITIME ?

In [ ]:
texte = "Offre : un cadeau pour vous"
total = sum(points(mot) for mot in mots_du_texte(texte))
print("total :", round(total, 1))

## Et celui-ci ?

« Les photos du club sont en ligne »

In [ ]:
texte = "Les photos du club sont en ligne"
for mot in mots_du_texte(texte):
    print(f"{mot:<8} {points(mot):+.1f}")
total = sum(points(mot) for mot in mots_du_texte(texte))
print("total :", round(total, 1))

Personne n'a écrit de règle sur « club » ou « photos ». Les exemples ont
suffi.

# À la main (exercices 5.1 et 5.2)

1. Remplissez le tableau 5.1 : gratuit, cliquez, votre, demain, réunion.
2. Avec ces points seulement, analysez les trois courriels.

| | Courriel |
|---|---|
| A | Cliquez ici : votre cadeau gratuit vous attend ! |
| B | Réunion demain : cliquez ici pour voir l'horaire. |
| C | Cliquez ici pour voir vos notes du cours. |

**Exercice 5.4 :** l'un des trois est mal classé. Lequel ? Quel mot l'a trompé ?

## Avec de vrais courriels

490 courriels pour apprendre, et trois catégories.

In [ ]:
from collections import Counter

courriels = pipeline.charger_courriels()
entrainement, validation = pipeline.separer_validation(courriels)
print(Counter(c["label"] for c in entrainement))

Les totaux ne sont plus égaux : on compare des **fréquences**.

`frequence = (compte + 1) / total`

## « gratuit » dans les vrais courriels

In [ ]:
def combien(mot):
    comptes = {"ham": 0, "spam": 0, "phishing": 0}
    for courriel in entrainement:
        if mot in mots_du_courriel(courriel):
            comptes[courriel["label"]] += 1
    return comptes


print(combien("gratuit"))

Les spams contre les **autres**. Combien de points pour
« gratuit » ?

- spams : 19 sur 147
- autres : 3 + 3 = 6 sur 196 + 147 = 343

In [ ]:
frequence_spam = (19 + 1) / 147
frequence_autres = (6 + 1) / 343
print(round(math.log2(frequence_spam / frequence_autres), 1))

## Ce que le module apprend vraiment

In [ ]:
print("ecole :", combien("ecole"))
print("qc    :", combien("qc"))

« ecole » vaut **+6.3** pour les courriels légitimes. C'est leur indice le
plus fort.

## Le compte de l'école est piraté

Un fraudeur vole le compte courriel de la direction. Que se passe-t-il ?

<img src="images/compte_pirate.svg" alt="" style="width:820px">

Notre module 3, une fois terminé, laisse passer ce courriel.
Il compte des mots, il ne comprend pas le message.

## Le test final

<img src="images/coffre_test.svg" alt="" style="width:720px">

Pourquoi ne faut-il jamais apprendre sur les courriels du test ?

Ce serait connaître les questions avant l'examen : une bonne note ne prouverait
rien. Le test final ne sert jamais à apprendre ni à régler.

# À vous

Fichier : `homework/bayes.py`

| Fonction | Exercice |
|---|---|
| `apprendre` | 5.1 |
| `points_du_mot` | 5.1 |
| `classer` | 5.2 |

Après chaque fonction, relancez la section **Vérification** de `main.ipynb`.
Ensuite, la section 6 montre les mots qui pèsent le plus pour **votre** module.